In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:\\Users\\DELL\\Documents\\Projects\\Churn Analysis\\Telco_customer_churn.csv")
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

In [5]:
# Convert TotalCharges to numeric
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

# Fill missing values
df['Total Charges'] = df['Total Charges'].fillna(df['Total Charges'].median())

In [6]:
df['Churn Label'] = df['Churn Label'].str.strip().str.title()

In [7]:
df['Churn Binary'] = df['Churn Label'].map({'Yes':1, 'No':0})

In [8]:
df['Churn Label'].value_counts(dropna=False)

Churn Label
No     5174
Yes    1869
Name: count, dtype: int64

In [10]:
df['RevenueRisk'] = df['Monthly Charges'] * df['Churn Binary']

In [11]:
df['TenureGroup'] = pd.cut(df['Tenure Months'],
                           bins=[0,12,24,48,72],
                           labels=['0-1yr','1-2yr','2-4yr','4-6yr'])

In [12]:
df['HighValue'] = np.where(df['Monthly Charges'] > df['Monthly Charges'].median(), 1, 0)

In [13]:
total_revenue = df['Monthly Charges'].sum()
churn_rate = df['Churn Binary'].mean()*100
revenue_at_risk = df[df['Churn Binary']==1]['Monthly Charges'].sum()

total_revenue, churn_rate, revenue_at_risk

(np.float64(456116.6), np.float64(26.536987079369588), np.float64(139130.85))

In [14]:
contract_churn = df.groupby('Contract')['Churn Binary'].mean()
internet_churn = df.groupby('Internet Service')['Churn Binary'].mean()
payment_churn = df.groupby('Payment Method')['Churn Binary'].mean()

print(contract_churn, internet_churn, payment_churn)

Contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
Name: Churn Binary, dtype: float64 Internet Service
DSL            0.189591
Fiber optic    0.418928
No             0.074050
Name: Churn Binary, dtype: float64 Payment Method
Bank transfer (automatic)    0.167098
Credit card (automatic)      0.152431
Electronic check             0.452854
Mailed check                 0.191067
Name: Churn Binary, dtype: float64


In [15]:
df.drop(columns=['Lat Long','Latitude','Longitude','Zip Code'], inplace=True)

In [16]:
df['RiskSegment'] = 'Low Risk'

df.loc[
    (df['Churn Binary']==1) &
    (df['Monthly Charges'] > df['Monthly Charges'].median()) &
    (df['Tenure Months'] < 12),
    'RiskSegment'
] = 'High Risk'

In [17]:
df.groupby('RiskSegment')['Monthly Charges'].sum()

RiskSegment
High Risk     45808.65
Low Risk     410307.95
Name: Monthly Charges, dtype: float64

In [19]:
df.groupby('TenureGroup')['Churn Binary'].mean()

C:\Users\DELL\AppData\Local\Temp\ipykernel_12300\2682001403.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('TenureGroup')['Churn Binary'].mean()


TenureGroup
0-1yr    0.476782
1-2yr    0.287109
2-4yr    0.203890
4-6yr    0.095132
Name: Churn Binary, dtype: float64

In [21]:
df.groupby('RiskSegment').agg({
    'CustomerID':'count',
    'Monthly Charges':'sum',
    'Churn Binary':'mean'
})

,CustomerID,Monthly Charges,Churn Binary
RiskSegment,,,
High Risk,544,45808.65,1.000000
Low Risk,6499,410307.95,0.203878


In [22]:
df[df['Churn Binary'] == 1][['CustomerID','Monthly Charges']].head()

,CustomerID,Monthly Charges
0,3668-QPYBK,53.85
1,9237-HQITU,70.70
2,9305-CDSKC,99.65
3,7892-POOKP,104.80
4,0280-XJGEX,103.70


In [25]:
df.to_csv(r"C:\\Users\\DELL\\Documents\\Projects\\Churn Analysis\\cleaned_churn_data1.csv", index=False)